In [ ]:
# If running in Google Colab, uncomment to upload the Phase-1 output file:
# from google.colab import files
# uploaded = files.upload()  # select Telco_Customer_Churn_Model_Ready.csv

!pip install imbalanced-learn shap -q


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              confusion_matrix, classification_report, average_precision_score)
from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")


## 1. Load Model-Ready Dataset (Phase 1 output)

In [ ]:
df = pd.read_csv("Telco_Customer_Churn_Model_Ready.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))


**Observation:** The dataset is imbalanced — approximately 26.5% of customers churned vs.
73.5% who did not. This is consistent with the ~26% baseline churn rate stated in the Phase 1
business problem. Because of this imbalance, accuracy alone is not a reliable metric; Precision,
Recall, F1-score, and ROC-AUC will be used for evaluation, and SMOTE will be applied to the
training set to help the models learn the minority (churn) class better.

## 2. Train / Test Split

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


## 3. Handle Class Imbalance with SMOTE (fit on training data only)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE :", y_train_res.value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train.value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Training Set - Before SMOTE")
y_train_res.value_counts().sort_index().plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Training Set - After SMOTE")
plt.tight_layout()
plt.show()


## 4. Hyperparameter Tuning
Both algorithms proposed in Phase 1 (Random Forest and Gradient Boosting) are tuned using
`RandomizedSearchCV` with 5-fold stratified cross-validation, optimizing ROC-AUC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_param_dist = {
    "n_estimators": [150, 250, 350, 450],
    "max_depth": [6, 8, 10, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced", None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_param_dist,
    n_iter=20, scoring="roc_auc", cv=cv, random_state=42, n_jobs=-1
)
rf_search.fit(X_train_res, y_train_res)
print("Best RF params:", rf_search.best_params_)
print("Best RF CV ROC-AUC:", rf_search.best_score_)
best_rf = rf_search.best_estimator_


In [ ]:
gb_param_dist = {
    "n_estimators": [100, 150, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "min_samples_leaf": [1, 5, 10, 20],
}

gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=20, scoring="roc_auc", cv=cv, random_state=42, n_jobs=-1
)
gb_search.fit(X_train_res, y_train_res)
print("Best GB params:", gb_search.best_params_)
print("Best GB CV ROC-AUC:", gb_search.best_score_)
best_gb = gb_search.best_estimator_


## 5. Evaluate on Held-Out Test Set

In [ ]:
def evaluate_model(model, name, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1-score": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
        "PR-AUC": average_precision_score(y_te, y_proba),
    }, y_pred, y_proba

rf_metrics, rf_pred, rf_proba = evaluate_model(best_rf, "Random Forest", X_test, y_test)
gb_metrics, gb_pred, gb_proba = evaluate_model(best_gb, "Gradient Boosting", X_test, y_test)

results_df = pd.DataFrame([rf_metrics, gb_metrics]).set_index("Model")
results_df.round(4)


In [ ]:
print("Random Forest\n", classification_report(y_test, rf_pred, target_names=["No Churn","Churn"]))
print("Gradient Boosting\n", classification_report(y_test, gb_pred, target_names=["No Churn","Churn"]))


## 6. Model Comparison Visualizations

In [ ]:
plot_df = results_df[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]].T
fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_title("Model Performance Comparison - Test Set")
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, proba, color in [("Random Forest", rf_proba, "#4C72B0"), ("Gradient Boosting", gb_proba, "#DD8452")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)
ax.plot([0,1],[0,1], linestyle="--", color="gray", label="Random Guess")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, proba, color in [("Random Forest", rf_proba, "#4C72B0"), ("Gradient Boosting", gb_proba, "#DD8452")]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})", color=color, linewidth=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve Comparison"); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, pred) in zip(axes, [("Random Forest", rf_pred), ("Gradient Boosting", gb_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Churn","Churn"], yticklabels=["No Churn","Churn"])
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()


## 7. Feature Importance & Business Interpretation

In [ ]:
best_model_name = results_df["ROC-AUC"].idxmax()
best_model = best_rf if best_model_name == "Random Forest" else best_gb
print("Best model by test ROC-AUC:", best_model_name)

importances = pd.Series(best_model.feature_importances_, index=feature_names)
top_features = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top_features.sort_values().plot(kind="barh", ax=ax, color="#55A868")
ax.set_title(f"Top 15 Feature Importances - {best_model_name}")
ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

top_features.head(10)


### Business Interpretation

- **Contract type** (month-to-month vs. one/two-year) is consistently the strongest driver of
  churn — customers without a long-term contract are far more likely to leave, confirming that
  contract lock-in is the single biggest lever for retention.
- **Monthly charges** and **tenure** are the next most important signals: newer customers on
  higher bills are highest risk, pointing to onboarding and early-tenure pricing/loyalty offers
  as a priority intervention window.
- **Add-on services** (Online Security, Tech Support, Online Backup) reduce churn risk —
  customers without these services churn more, suggesting bundling or promoting these services
  could improve stickiness.
- These findings align directly with the Phase 1 KPI targets: the model's churn-probability
  score can be used to target the top 20% at-risk customers (Retention Campaign ROI KPI) and
  prioritize contract-renewal and add-on-service offers before contract end dates (Revenue
  Leakage KPI).


## 8. Save Final Results

In [ ]:
results_df.round(4).to_csv("model_comparison_results.csv")
print("Saved model_comparison_results.csv")
results_df.round(4)
